# Fault-aware training and reliability comparison

This notebook asks whether an occupancy model becomes safer when simulated Light failures are included during training. It compares the existing three-sensor primary, causal detector routing, fault-aware retraining, and fault-aware retraining with an explicit missing-Light indicator.

The models are trained by packaged Python code, not by notebook cells. This notebook loads the saved results and explains how the training-only selection leads to the held-out conclusion.

## 1. Experimental design

Each chronological development fold keeps every clean row and adds a small sample of faulty Light rows. The search compares 1%, 5%, and 10% augmentation; four algorithms; and representations with or without `Light_missing`.

The clean-trained logistic primary supplies a guardrail: a candidate may lose at most 0.02 mean clean-validation F1. Test 1 and Test 2 are not used to choose the model, augmentation strength, or representation.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Allow the notebook to run from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULT_DIR = PROJECT_ROOT / "models" / "fault_aware"
required = {
    "cv": RESULT_DIR / "cv_summary.csv",
    "heldout": RESULT_DIR / "heldout_metrics.csv",
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run `python -m sensorbudget.robustness.fault_aware` first. "
        + f"Missing: {missing}"
    )

cv = pd.read_csv(required["cv"])
heldout = pd.read_csv(required["heldout"])
PLOTLY_TEMPLATE = "plotly_white"
SPLIT_LABELS = {"test_1": "Test 1", "test_2": "Test 2"}
STRATEGY_LABELS = {
    "primary_only": "Primary only",
    "detector_routing": "Detector routing",
    "fault_aware": "Fault-aware (diagnostic)",
    "fault_aware_missing_indicator": "Fault-aware + missing indicator",
}
SCENARIO_LABELS = {
    "clean": "Clean data",
    "missing": "Missing Light",
    "stuck_current": "Frozen at current value",
    "stuck_low": "Fixed low",
    "stuck_high": "Fixed high",
    "out_of_range_high": "Above training range",
    "linear_bias_positive": "Positive calibration bias",
    "linear_bias_negative": "Negative calibration bias",
}
heldout["split_label"] = heldout["split"].map(SPLIT_LABELS)
heldout["strategy_label"] = heldout["strategy"].map(STRATEGY_LABELS)
heldout["scenario_label"] = heldout["scenario"].map(SCENARIO_LABELS)
print(f"Loaded {len(cv)} candidates and {len(heldout)} held-out evaluations.")

## 2. Which candidates were retained?

A selected row is the candidate carried forward for that representation. The table records exactly what inputs it receives, how it treats missing Light, and whether it passed the clean-F1 guardrail. A rejected candidate can still be retained as a diagnostic comparison so that failure is visible rather than discarded.

In [ ]:
# Show both the implementation difference and its validation outcome.
selected = cv.loc[cv["selected"]].copy()
selected["representation"] = selected["representation"].map(STRATEGY_LABELS)
selected["Model inputs"] = selected["representation"].map({
    "Fault-aware (diagnostic)": "Temperature, Light, CO₂",
    "Fault-aware + missing indicator": "Temperature, Light, CO₂, Light_missing",
})
selected["Missing-Light handling"] = selected["representation"].map({
    "Fault-aware (diagnostic)": (
        "Replace Light with the training median; the model is not told "
        "that replacement occurred"
    ),
    "Fault-aware + missing indicator": (
        "Replace Light with the training median and set Light_missing = 1"
    ),
})
selected["Selection status"] = selected["guardrail_satisfied"].map({
    True: "Accepted by training validation",
    False: "Diagnostic only — narrowly missed clean-F1 guardrail",
})
display_table = selected.rename(columns={
    "representation": "Approach",
    "model": "Algorithm",
    "augmentation_ratio": "Added fault rows",
    "clean_reference_f1_mean": "Reference clean F1",
    "clean_f1_mean": "Candidate clean F1",
    "fault_f1_mean": "Mean occupancy F1 under faults",
})
display(
    display_table[[
        "Approach", "Model inputs", "Missing-Light handling",
        "Algorithm", "Added fault rows", "Reference clean F1",
        "Candidate clean F1", "Mean occupancy F1 under faults",
        "Selection status",
    ]].style.format({
        "Added fault rows": "{:.0%}",
        "Reference clean F1": "{:.3f}",
        "Candidate clean F1": "{:.3f}",
        "Mean occupancy F1 under faults": "{:.3f}",
    }).set_properties(**{"white-space": "normal", "text-align": "left"})
)

**Conclusion.** Both representations retain logistic regression with 1% augmentation. The missing-indicator version passes the validation guardrail. The plain version misses it by a very small margin and is therefore labelled diagnostic rather than accepted.

## 3. Occupancy performance on clean and faulted validation data

Within each chronological fold, the model is fitted once on the clean development rows plus its small fault-augmented sample. That same fitted model is then evaluated in two stages:

1. **Clean validation F1:** occupancy predictions on the untouched validation period.
2. **Mean occupancy F1 under simulated Light faults:** occupancy predictions on seven separate copies of that validation period, with one whole-period Light fault applied to each copy.

The second metric is classification performance after sensor corruption; it is not the detector's ability to identify faults. The dotted line is the minimum permitted clean score.

In [ ]:
# Plot the retained candidates horizontally for readable strategy names.
plot_rows = selected.copy()
fig = go.Figure()
# Draw the fault result first so clean validation appears above it in each
# horizontal group. Reverse the legend to match the visible bar order.
for metric, label, color in [
    ("fault_f1_mean", "Mean occupancy F1 under Light faults", "#e76f51"),
    ("clean_f1_mean", "Clean validation F1", "#457b9d"),
]:
    fig.add_bar(
        x=plot_rows[metric], y=plot_rows["representation"],
        orientation="h", name=label, marker_color=color,
        text=plot_rows[metric].map(lambda value: f"{value:.3f}"),
        textposition="outside",
        hovertemplate=f"%{{y}}<br>{label}: %{{x:.3f}}<extra></extra>",
    )
clean_floor = float(
    selected["clean_reference_f1_mean"].iloc[0] - 0.02
)
fig.add_vline(
    x=clean_floor, line_dash="dot", line_color="#555555",
    annotation_text="Clean guardrail", annotation_position="top",
)
fig.update_layout(
    template=PLOTLY_TEMPLATE, barmode="group",
    title="Occupancy F1 on clean and faulted chronological validation data",
    legend={"traceorder": "reversed"},
    xaxis={"title": "Mean chronological-validation F1", "range": [0, 1.01]},
    yaxis_title=None, height=430, margin={"l": 210, "r": 40},
)
fig.show()

**Conclusion.** Fault augmentation does not make the model uniformly robust. Mean F1 under full-period validation faults remains around 0.432, showing that plausible stuck and drifting signals are still difficult even when examples are injected during training.

## 4. What does retraining cost on pristine held-out data?

This is the most important safety check. No fault is injected, so any difference is the cost of changing the training procedure or unnecessarily routing healthy observations.

In [ ]:
clean = heldout.loc[heldout["scenario"] == "clean"].copy()
strategy_order = list(STRATEGY_LABELS.values())
fig = go.Figure()
for split, color in [("Test 2", "#e76f51"), ("Test 1", "#457b9d")]:
    rows = clean.loc[clean["split_label"] == split].copy()
    rows["strategy_label"] = pd.Categorical(
        rows["strategy_label"], categories=strategy_order, ordered=True
    )
    rows = rows.sort_values("strategy_label", ascending=False)
    fig.add_bar(
        x=rows["f1"], y=rows["strategy_label"], orientation="h",
        name=split, marker_color=color,
        text=rows["f1"].map(lambda value: f"{value:.3f}"),
        textposition="outside",
        hovertemplate="%{y}<br>F1: %{x:.3f}<extra></extra>",
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE, barmode="group",
    title="Pristine held-out F1", xaxis={"title": "F1", "range": [0, 1.01]},
    legend={"traceorder": "reversed"},
    yaxis_title=None, height=500, margin={"l": 230, "r": 40},
)
fig.show()

**Conclusion.** Test 1 does not reveal a clean-data penalty, but Test 2 does. The missing-indicator model falls from 0.980 to 0.942, while the plain diagnostic model falls to 0.917. Passing training validation therefore did not guarantee later-period stability.

## 5. Average performance across all tested conditions

The bars average clean data and every 15- and 60-row fault episode. This summary is useful for comparison, but it does not imply that all failures are equally likely in deployment.

In [ ]:
average = (
    heldout.groupby(["split_label", "strategy_label"], as_index=False)["f1"]
    .mean()
)
fig = go.Figure()
for split, color in [("Test 2", "#e76f51"), ("Test 1", "#457b9d")]:
    rows = average.loc[average["split_label"] == split].copy()
    rows["strategy_label"] = pd.Categorical(
        rows["strategy_label"], categories=strategy_order, ordered=True
    )
    rows = rows.sort_values("strategy_label", ascending=False)
    fig.add_bar(
        x=rows["f1"], y=rows["strategy_label"], orientation="h",
        name=split, marker_color=color,
        text=rows["f1"].map(lambda value: f"{value:.3f}"),
        textposition="outside",
        hovertemplate="%{y}<br>Mean F1: %{x:.3f}<extra></extra>",
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE, barmode="group",
    title="Mean F1 across clean and episodic Light-fault cases",
    legend={"traceorder": "reversed"},
    xaxis={"title": "Mean F1", "range": [0, 1.01]}, yaxis_title=None,
    height=500, margin={"l": 230, "r": 40},
)
fig.show()

**Conclusion.** Detector routing has the highest mean F1 on Test 1 at approximately 0.970. On Test 2, the unchanged primary is slightly higher (0.978 versus 0.974 for detector routing), because short injected episodes often leave full-period primary performance high while detector false alarms can route healthy rows. Among the tested mitigation alternatives, detector routing remains strongest overall; the missing-indicator model averages only 0.939 on Test 2.

## 6. Phase 5 reliability conclusion

The tested fault-aware models are not accepted as replacements for the current primary-plus-detector-routing design. A missingness indicator helps when Light is explicitly absent, but it cannot identify present-yet-wrong readings such as plausible darkness or gradual bias. More importantly, retraining creates a material pristine Test 2 penalty.

Detector routing is the strongest tested mitigation overall, but it still misses plausible faults and occasionally routes healthy rows because of false alarms. This is a comparative project conclusion, not a deployment certification or a final sensor recommendation.